In [2]:
import pandas as pd
import numpy as np

portfolio = pd.read_csv("../data/Portfolio_clean.csv")
profile = pd.read_csv("../data/profile_clean.csv")
menu = pd.read_csv("../data/starbucks_menu_clean.csv")
transcript = pd.read_csv("../data/transcript_clean.csv")
transcript_flat = pd.read_csv("../data/transcript_cleaned_flattened.csv")

In [3]:
df = transcript_flat
df = df.rename(columns={
    "amount value": "amount",
    "time": "event_time"
})

In [4]:
df["event_time"] = df["event_time"].astype(int)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

In [5]:
received = df[df["event"] == "offer received"].copy()
viewed = df[df["event"] == "offer viewed"].copy()
completed = df[df["event"] == "offer completed"].copy()
transactions = df[df["event"] == "transaction"].copy()

In [6]:
# Each “offer received” = one instance
received = received.sort_values(["customer_id", "offer_id", "event_time"])

received["instance_id"] = (
    received.groupby(["customer_id", "offer_id"])
    .cumcount() + 1
)

In [7]:
# Join Portfolio (difficulty + duration)
portfolio = portfolio.rename(columns={
    "promo_id": "offer_id"
})

received = received.merge(
    portfolio[["offer_id", "difficulty", "duration"]],
    on="offer_id",
    how="left"
)

# convert duration to hours (Starbucks uses days → hours)
received["expiry_time"] = received["event_time"] + received["duration"] * 24

In [8]:
# keep only needed columns
received_small = received[["customer_id", "offer_id", "instance_id", "event_time", "expiry_time"]].copy()
viewed_small = viewed[["customer_id", "offer_id", "event_time"]].copy()

# rename times so they don't collide
received_small = received_small.rename(columns={"event_time": "received_time"})
viewed_small = viewed_small.rename(columns={"event_time": "viewed_time"})

# merge on customer_id + offer_id
rv = received_small.merge(
    viewed_small,
    on=["customer_id", "offer_id"],
    how="left"
)

# keep only viewed events inside valid window
rv = rv[
    (rv["viewed_time"] >= rv["received_time"]) &
    (rv["viewed_time"] <= rv["expiry_time"])
]

# for each received instance, take the earliest valid viewed_time
first_viewed = (
    rv.groupby(["customer_id", "offer_id", "instance_id"], as_index=False)["viewed_time"]
    .min()
)

# merge back into received
received = received.merge(
    first_viewed,
    on=["customer_id", "offer_id", "instance_id"],
    how="left"
)

In [9]:
# keep only what we need
received_tx = received[[
    "customer_id", "offer_id", "instance_id",
    "viewed_time", "expiry_time"
]].copy()

tx_small = transactions[[
    "customer_id", "event_time", "amount"
]].copy().rename(columns={"event_time": "txn_time"})

# merge transactions to each customer's offer instance
rt = received_tx.merge(
    tx_small,
    on="customer_id",
    how="left"
)

# keep only transactions after viewed and before expiry
rt = rt[
    rt["viewed_time"].notna() &
    (rt["txn_time"] >= rt["viewed_time"]) &
    (rt["txn_time"] <= rt["expiry_time"])
]

# sum transaction amount per offer instance
txn_sum = (
    rt.groupby(["customer_id", "offer_id", "instance_id"], as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "txn_sum"})
)

# merge back
received = received.merge(
    txn_sum,
    on=["customer_id", "offer_id", "instance_id"],
    how="left"
)

received["txn_sum"] = received["txn_sum"].fillna(0)

In [10]:
received["txn_satisfied"] = received["txn_sum"] >= received["difficulty"]

In [11]:
# keep only needed columns
received_cp = received[[
    "customer_id", "offer_id", "instance_id",
    "viewed_time", "expiry_time"
]].copy()

completed_small = completed[[
    "customer_id", "offer_id", "event_time"
]].copy().rename(columns={"event_time": "completed_time"})

# merge on customer_id + offer_id
rc = received_cp.merge(
    completed_small,
    on=["customer_id", "offer_id"],
    how="left"
)

# keep only completed events after viewed and before expiry
rc = rc[
    rc["viewed_time"].notna() &
    (rc["completed_time"] >= rc["viewed_time"]) &
    (rc["completed_time"] <= rc["expiry_time"])
]

# earliest valid completed per offer instance
first_completed = (
    rc.groupby(["customer_id", "offer_id", "instance_id"], as_index=False)["completed_time"]
    .min()
)

# merge back into received
received = received.merge(
    first_completed,
    on=["customer_id", "offer_id", "instance_id"],
    how="left"
)

In [12]:
received["valid_completion"] = (
    (received["viewed_time"].notna()) &
    (received["txn_satisfied"]) &
    (received["completed_time"].notna())
)

In [13]:
result = received[[
    "customer_id",
    "offer_id",
    "instance_id",
    "event_time",
    "viewed_time",
    "expiry_time",
    "txn_sum",
    "difficulty",
    "txn_satisfied",
    "completed_time",
    "valid_completion"
]]

display(result.head(20))
result[result["customer_id"]=="003d66b6608740288d6cc97a6903f4f0"]

,customer_id,offer_id,instance_id,event_time,viewed_time,expiry_time,txn_sum,difficulty,txn_satisfied,completed_time,valid_completion
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,1,576,NaN,744,0.00,10,False,NaN,False
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,1,336,372.0,432,8.57,0,True,NaN,False
2,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,1,168,192.0,240,22.16,0,True,NaN,False
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,1,408,456.0,528,14.11,5,True,NaN,False
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,1,504,540.0,744,82.76,10,True,NaN,False
5,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,1,168,216.0,288,0.00,5,False,NaN,False
6,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,2,576,630.0,696,0.00,5,False,NaN,False
7,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,408,432.0,648,45.08,20,True,576.0,True
8,0011e0d4e6b944f998e987f904e8c1e5,2298d6c36e964ae4a3e7e9706d1fb8c2,1,168,186.0,336,11.93,7,True,252.0,True
9,0011e0d4e6b944f998e987f904e8c1e5,3f207df678b143eea3cee63160fa8bed,1,0,6.0,96,0.00,0,True,NaN,False


,customer_id,offer_id,instance_id,event_time,viewed_time,expiry_time,txn_sum,difficulty,txn_satisfied,completed_time,valid_completion
21,003d66b6608740288d6cc97a6903f4f0,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,504,NaN,744,0.00,20,False,NaN,False
22,003d66b6608740288d6cc97a6903f4f0,3f207df678b143eea3cee63160fa8bed,1,336,372.0,432,10.08,0,True,NaN,False
23,003d66b6608740288d6cc97a6903f4f0,5a8bc65990b245e5a138643cd4eb9837,1,0,36.0,72,2.51,0,True,NaN,False
24,003d66b6608740288d6cc97a6903f4f0,fafdcd668e3743c1bb461111dcafc2a4,1,168,300.0,408,14.97,10,True,384.0,True
25,003d66b6608740288d6cc97a6903f4f0,fafdcd668e3743c1bb461111dcafc2a4,2,408,420.0,648,24.45,10,True,504.0,True


## 충성고객에 대한 코드
프로모션 기간에 viewed 없이도 complete를 수행한 사람.

In [15]:
# find completed within receive~expiry window, regardless of viewed_time
received_cp2 = received[[
    "customer_id", "offer_id", "instance_id",
    "event_time", "expiry_time"
]].copy().rename(columns={"event_time": "received_time"})

completed_small = completed[[
    "customer_id", "offer_id", "event_time"
]].copy().rename(columns={"event_time": "completed_time_no_view_check"})

rc2 = received_cp2.merge(
    completed_small,
    on=["customer_id", "offer_id"],
    how="left"
)

rc2 = rc2[
    (rc2["completed_time_no_view_check"] >= rc2["received_time"]) &
    (rc2["completed_time_no_view_check"] <= rc2["expiry_time"])
]

first_completed_no_view = (
    rc2.groupby(["customer_id", "offer_id", "instance_id"], as_index=False)["completed_time_no_view_check"]
    .min()
)

received = received.merge(
    first_completed_no_view,
    on=["customer_id", "offer_id", "instance_id"],
    how="left"
)

In [16]:
loyal = received[
    (received["viewed_time"].isna()) &
    (received["completed_time_no_view_check"].notna())
]

num_loyal_customers = loyal["customer_id"].nunique()

print("Number of customers with no viewed event but completed within duration:", num_loyal_customers)
print("Total such offer instances:", loyal.shape[0])

Number of customers with no viewed event but completed within duration: 4460
Total such offer instances: 5734


In [19]:
# get customer sets
promo_valid = set(
    received[received["valid_completion"] == True]["customer_id"]
)

loyal_valid = set(
    received[
        (received["viewed_time"].isna()) &
        (received["completed_time_no_view_check"].notna())
    ]["customer_id"]
)

# intersection
overlap = promo_valid & loyal_valid

print("Number of overlapping customers:", len(overlap))

Number of overlapping customers: 3596


In [20]:
print("Sample overlapping customers:", list(overlap)[:10])

Sample overlapping customers: ['df3da7cfcf614b3481b65c89657994ed', '07bdb81f215b4ef19653675c6eb2c447', '227ce6b51df94aeaa83ce2701eb2963c', 'f838af6aec284aad99aee4039f8eb9cf', 'f586fb1597464f8b83982ea685f35b28', '66ad7d1394f24c789a3ea768727bf07b', '2cb4d7a40765483cb808ce28107a3775', '93e2bbb847db4cb7a082b1ecd2568e21', 'b56e747d7f6641c88cacc52c5eaebf57', 'd61124b6a76847b593f8fce78484fcce']
